In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split
import pandas as pd

data = sns.load_dataset('titanic')

# 학습용 데이터와 정답을 분리
# 훈련 검증 테스트 데이터 분리
# PyTorch로 신경망 구성
# 손실함수는 BCELoss
# optimizer adam, lr=기본값

X = data.drop(columns =[ 'survived','class','who','adult_male','deck','embark_town','alive' ])
y = data['survived']

X = pd.get_dummies(X, columns=['sex'])

x_train_full,x_test,y_train_full,y_test = train_test_split(X, y, test_size=0.2, random_state=42)
x_train,x_valid,y_train,y_valid = train_test_split(x_train_full, y_train_full, test_size=0.25, random_state=42)

# import torch
# import torch.nn as nn
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()
# x_train = scaler.fit_transform(x_train)
# x_valid = scaler.transform(x_valid)
# x_test = scaler.transform(x_test)

# x_train = torch.tensor(x_train, dtype=torch.float32)
# y_train = torch.tensor(y_train, dtype=torch.float32)

# class MLP(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(14, 60),
#             nn.ReLU(),
#             nn.Linear(60, 40),
#             nn.ReLU(),
#             nn.Linear(40, 1),
#             nn.Sigmoid()
#         )

#     def forward(self, x):
#         return self.net(x)

# model = MLP()
# print(model)


In [15]:
X

,pclass,age,sibsp,parch,fare,embarked,alone,sex_female,sex_male
0,3,22.0,1,0,7.2500,S,False,False,True
1,1,38.0,1,0,71.2833,C,False,True,False
2,3,26.0,0,0,7.9250,S,True,True,False
3,1,35.0,1,0,53.1000,S,False,True,False
4,3,35.0,0,0,8.0500,S,True,False,True
...,...,...,...,...,...,...,...,...,...
886,2,27.0,0,0,13.0000,S,True,False,True
887,1,19.0,0,0,30.0000,S,True,True,False
888,3,NaN,1,2,23.4500,S,False,True,False
889,1,26.0,0,0,30.0000,C,True,False,True


In [138]:
# 수치형 standard.... 범주형 OneHot
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.impute  import SimpleImputer
import torch.optim as optim

titanic = sns.load_dataset('titanic')
y = titanic['survived'].to_numpy()
X = titanic.loc[:,'pclass':]
X.drop(columns=['class','sex','embark_town','alive','deck'])
numeric_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare']
categorical_cols = ['embarked', 'who', 'adult_male', 'alone']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, numeric_cols),
    ('cat', cat_pipeline, categorical_cols)
])

x_train_full,x_test,y_train_full,y_test = train_test_split(X, y, test_size = 0.2, random_state=42, stratify = y)
x_train,x_valid,y_train,y_valid = train_test_split(x_train_full, y_train_full, test_size=0.25, random_state=42, stratify = y_train_full)

x_train = preprocessor.fit_transform(x_train)
x_valid =preprocessor.transform(x_valid)
x_test = preprocessor.transform(x_test)




In [139]:
import torch.nn as nn
import torch

x_train_t = torch.tensor( x_train, dtype=torch.float32 )
x_valid_t = torch.tensor( x_valid, dtype=torch.float32 )
x_test_t = torch.tensor( x_test, dtype=torch.float32) 

y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model = MLP(input_dim=x_train_t.shape[1])

    

In [141]:
creterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.1)

for epoch in range(101):
    y_hat = model(x_train_t)
    loss = creterion(y_hat, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        pred = (y_hat >= 0.5).float()   # 0.5를 기준으로 0 또는 1
        acc = (pred == y_train_t).float().mean()   ## 정확도 계산
        print(f"Epoch {epoch:5d} | Loss: {loss.item():.4f} | Acc: {acc:.2f}")

print("\n예측 결과:")
with torch.no_grad():
    # no_grad: 기울기 계산 안 함
    # 예측할 때는 역전파 필요 X
    # 메모리 절약 + 속도 향상
    for i in range(4):
        pred = model(x_train_t[i:i+1])
        print(f"{x_train_t[i].tolist()} -> {pred.item():.4f} (정답 : {y_train_t[i].item()})")

Epoch     0 | Loss: 0.2692 | Acc: 0.89
Epoch    10 | Loss: 0.3653 | Acc: 0.85
Epoch    20 | Loss: 0.3327 | Acc: 0.87
Epoch    30 | Loss: 0.3006 | Acc: 0.88
Epoch    40 | Loss: 0.2776 | Acc: 0.89
Epoch    50 | Loss: 0.2527 | Acc: 0.89
Epoch    60 | Loss: 0.2338 | Acc: 0.90
Epoch    70 | Loss: 0.2207 | Acc: 0.90
Epoch    80 | Loss: 0.2142 | Acc: 0.90
Epoch    90 | Loss: 0.3091 | Acc: 0.90
Epoch   100 | Loss: 0.2370 | Acc: 0.90

예측 결과:
[-1.5710841417312622, 0.6772592067718506, 0.4684527516365051, -0.4717281758785248, 0.534936785697937, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0] -> 1.0000 (정답 : 1.0)
[0.8328096270561218, -1.9936842918395996, 2.3283321857452393, 1.8607056140899658, -0.07261662930250168, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0] -> 0.0177 (정답 : 0.0)
[0.8328096270561218, -0.9253068566322327, 0.4684527516365051, 0.6944887042045593, -0.23942258954048157, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0] -> 0.0000 (정답 : 0.0)
[-1.5710841417312622, 1.8219492435455322